In [ ]:
# Copyright (c) 2024-2026 ORIQX AG/LTD/SAS. All rights reserved.
# PROPRIETARY AND CONFIDENTIAL. Unauthorized copying, distribution,
# or use of this file, via any medium, is strictly prohibited.

# Classical-Quantum Interaction: The Variational Loop Pattern

**A comprehensive guide to writing hybrid quantum-classical algorithms on uniqx.**

Most near-term quantum algorithms are **hybrid**: a classical computer runs an
optimisation loop that repeatedly calls a quantum processor (or simulator) to
evaluate a cost function. The two most important examples are:

- **VQE** (Variational Quantum Eigensolver) — finds molecular ground-state energies.
- **QAOA** (Quantum Approximate Optimisation Algorithm) — solves combinatorial problems.

Both follow the same **variational loop** pattern:

```
classical parameters theta
 |
 v
 [Prepare quantum state |psi(theta)>] <-- server-side: ops.expv
 |
 v
 [Measure cost function <psi|H|psi>] <-- server-side: ops.expect
 |
 v
 classical optimizer updates theta
 |
 v
 repeat until converged
```

### Key design principle

**All quantum computation is server-side.** The classical optimiser lives in Python,
but every quantum operation (`expv`, `expect`, `eigs`) is traced into the IR and
executed on uniqx. uniqx routes each primitive to the best available
hardware:

| Primitive | Classical backend | Quantum backend |
|:----------|:------------------|:----------------|
| `ops.expv(H, psi, t)` | Dense matrix exponential (CPU/GPU) | Trotterised circuit (QPU) |
| `ops.expect(O, psi)` | $\psi^\dagger O \psi$ (CPU/GPU) | Pauli measurement (QPU) |
| `ops.eigs(H, k=1)` | LAPACK / Lanczos (CPU/GPU) | QPE / VQE (QPU) |

This notebook walks through the full cycle: preparing circuits, submitting them,
parsing results, and feeding them back into a classical optimiser.

---
## 1. Setup and Imports

In [ ]:
import os
from uniqx import connect, login

endpoint = os.environ.get("UNIQX_GATEWAY", "api.oriqx.com:443")
login(os.environ["UNIQX_API_KEY"], gateway=endpoint)
client = connect(endpoint)
print("Connected to", endpoint)
import math
import numpy as np
import matplotlib.pyplot as plt

# uniqx SDK imports
from uniqx import ops, tracing
from uniqx.ops import SX, SY, SZ, I2
from uniqx.dialects import quantum
from uniqx.core.execution import (
    connect, submit, get, preflight,
    fmt_mat, fmt_vec, fmt_scalar,
    parse_buffer_view, parse_result,
)

# Connect to uniqx
API_KEY = os.environ.get("UNIQX_API_KEY")


---
## 2. Circuit Preparation: Building Parameterised Quantum Circuits

In uniqx, quantum circuits are built by composing **traced operations** inside a
`@tracing.to_module` function. The key primitives are:

- **`ops.embed(op, qubit, n_qubits)`**: Embed a single-qubit operator (e.g. a
 rotation matrix) into the full $2^n \times 2^n$ Hilbert space via Kronecker
 products: $I \otimes \cdots \otimes \text{op} \otimes \cdots \otimes I$.

- **`ops.two_body(opA, opB, qa, qb, n_qubits)`**: Embed a two-qubit interaction
 term $\text{opA} \otimes \text{opB}$ at qubits `qa` and `qb`.

- **`ops.expv(A, v, t)`**: Compute $\exp(tA)\,v$. This is the workhorse for
 state preparation — applying a parameterised unitary to a state vector.

- **`ops.expect(O, state)`**: Compute $\langle\text{state}|\,O\,|\text{state}\rangle$.
 This measures the cost function.

Parameters flow from the classical optimiser as **runtime inputs** to the traced
module. The module is traced once (defining the computation graph), then executed
repeatedly with different parameter values.

### 2.1 Single-qubit rotations via `ops.embed`

A rotation $R_y(\theta)$ around the Y-axis for qubit $k$ in an $n$-qubit system
is implemented as:

$$R_y^{(k)}(\theta) = I \otimes \cdots \otimes R_y(\theta) \otimes \cdots \otimes I$$

where $R_y(\theta) = \exp(-i\theta\sigma_y/2)$. In uniqx, the rotation matrix
is a $2\times2$ Python list, and `ops.embed` handles the Kronecker chain.

In [ ]:
def ry_matrix(theta):
    """Build the 2x2 Ry(theta) rotation matrix."""
    c = math.cos(theta / 2)
    s = math.sin(theta / 2)
    return [[c, -s], [s, c]]


def cnot_matrix():
    """Build the 4x4 CNOT matrix (control=0, target=1)."""
    return [
        [1.0, 0.0, 0.0, 0.0],
        [0.0, 1.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 1.0],
        [0.0, 0.0, 1.0, 0.0],
    ]


# Example: embed Ry(pi/4) on qubit 0 of a 2-qubit system
theta_example = math.pi / 4
ry_mat = ry_matrix(theta_example)
print(f"Ry({theta_example:.4f}) matrix:")
for row in ry_mat:
    print(f"  [{row[0]:+.4f}, {row[1]:+.4f}]")

### 2.2 Two-qubit entangling gates via `ops.two_body`

For entangling gates like CNOT, we use `ops.matmul` with the full $4\times4$
matrix, or decompose into two-body Pauli terms. The CNOT gate creates
entanglement between qubits, which is essential for capturing quantum correlations
in the ansatz.

### 2.3 The traced module pattern

Every quantum computation must be wrapped in `@tracing.to_module`. This decorator
traces the Python function into an IR graph (like a computation DAG) that the
uniqx can compile and execute.

```python
@tracing.to_module
def my_circuit(hamiltonian, initial_state):
 # ops inside here are traced, not executed immediately
 evolved = ops.expv(hamiltonian, initial_state, -1.0, hermitian=True)
 energy = ops.expect(hamiltonian, evolved)
 return energy

# Tracing: creates the IR module (no execution yet)
module = my_circuit(H_matrix, psi0_vector)

# Execution: send to uniqx with actual numeric values
job_id = submit(module, client=client, runtime_inputs=[H_values, psi0_values])
result = get(job_id, client=client)
```

The key insight: **tracing defines the shape of computation; runtime inputs
provide the actual numbers.** This separation lets you trace once and execute
many times with different parameters.

---
## 3. VQE Example: H$_2$ Ground State Energy

We solve for the ground state of the hydrogen molecule (H$_2$) in a minimal
STO-3G basis. The Jordan-Wigner transformation maps this to a **2-qubit**
Hamiltonian:

$$H = g_0 I + g_1 Z_0 + g_2 Z_1 + g_3 Z_0 Z_1 + g_4 X_0 X_1 + g_5 Y_0 Y_1$$

At equilibrium bond length (0.735 Angstrom), the coefficients are:

| Coefficient | Value |
|:------------|:------|
| $g_0$ | $-0.4804$ |
| $g_1$ | $+0.3435$ |
| $g_2$ | $-0.4347$ |
| $g_3$ | $+0.5716$ |
| $g_4$ | $+0.0910$ |
| $g_5$ | $+0.0910$ |

The exact ground-state energy is approximately **$-1.137$ Hartree**.

### 3.1 Build the H$_2$ Hamiltonian

In [ ]:
# H2 Hamiltonian coefficients (STO-3G, bond length 0.735 A)
g0 = -0.4804
g1 = +0.3435
g2 = -0.4347
g3 = +0.5716
g4 = +0.0910
g5 = +0.0910

# Pauli matrices (spin-1/2 operators, sigma/2)
# SX, SY, SZ, I2 are provided by uniqx.ops
# Full Pauli matrices (without the 1/2 factor)
pauli_X = [[0.0, 1.0], [1.0, 0.0]]
pauli_Y = [[0.0, -1.0], [1.0, 0.0]]  # real part; imag cancels in YY
pauli_Z = [[1.0, 0.0], [0.0, -1.0]]
identity = [[1.0, 0.0], [0.0, 1.0]]


def kron_2x2(a, b):
    """Kronecker product of two 2x2 matrices -> 4x4."""
    result = [[0.0] * 4 for _ in range(4)]
    for i in range(2):
        for j in range(2):
            for k in range(2):
                for l in range(2):
                    result[2 * i + k][2 * j + l] = a[i][j] * b[k][l]
    return result


def add_mat(a, b, scale_b=1.0):
    """Add two 4x4 matrices: a + scale_b * b."""
    n = len(a)
    return [[a[i][j] + scale_b * b[i][j] for j in range(n)] for i in range(n)]


def scale_mat(s, m):
    """Scale a matrix by scalar s."""
    n = len(m)
    return [[s * m[i][j] for j in range(n)] for i in range(n)]


# Build H = g0*II + g1*ZI + g2*IZ + g3*ZZ + g4*XX + g5*YY
H_II = kron_2x2(identity, identity)
H_ZI = kron_2x2(pauli_Z, identity)
H_IZ = kron_2x2(identity, pauli_Z)
H_ZZ = kron_2x2(pauli_Z, pauli_Z)
H_XX = kron_2x2(pauli_X, pauli_X)
H_YY = kron_2x2(pauli_Y, pauli_Y)

H2_hamiltonian = [[0.0] * 4 for _ in range(4)]
for term, coeff in [(H_II, g0), (H_ZI, g1), (H_IZ, g2),
                     (H_ZZ, g3), (H_XX, g4), (H_YY, g5)]:
    H2_hamiltonian = add_mat(H2_hamiltonian, term, coeff)

print("H2 Hamiltonian (4x4):")
for row in H2_hamiltonian:
    print("  [" + ", ".join(f"{v:+.4f}" for v in row) + "]")

### 3.2 Exact diagonalisation (reference)

First, compute the exact ground-state energy using `ops.eigs`. This gives us
the target value that VQE should converge to.

In [ ]:
@tracing.to_module
def exact_diag(H):
    """Compute the smallest eigenvalue of H."""
    eigvals, eigvecs = ops.eigs(H, k=1, which="smallest", hermitian=True)
    return eigvals


# Trace and submit (with guard against backend limitations)
try:
    mod_exact = exact_diag(H2_hamiltonian)
    jid = submit(mod_exact, client=client, runtime_inputs=[H2_hamiltonian])
    res = get(jid, client=client)

    parsed = parse_result(res.get("result_payload", b""), ["eigvals"])
    exact_energy = parsed["eigvals"][2][0]  # (dims, dtype, values) -> values[0]
    print(f"Exact ground-state energy: {exact_energy:.6f} Ha")
    print(f"Expected (literature):     -1.137 Ha")
except (KeyError, IndexError, RuntimeError) as e:
    print(f"Known limitation: eigs on this matrix shape not yet supported ({type(e).__name__})")
    exact_energy = -1.137  # fall back to literature value for downstream display
    print(f"Using literature reference: {exact_energy:.6f} Ha")


### 3.3 Hardware-efficient ansatz

The VQE ansatz is a parameterised quantum circuit that prepares trial states.
We use a **hardware-efficient** ansatz:

$$|\psi(\boldsymbol{\theta})\rangle = \text{CNOT}_{01} \cdot R_y^{(1)}(\theta_3) \cdot R_y^{(0)}(\theta_2) \cdot \text{CNOT}_{01} \cdot R_y^{(1)}(\theta_1) \cdot R_y^{(0)}(\theta_0) \;|00\rangle$$

This is implemented using `ops.expv` to apply each unitary layer to the state
vector, or equivalently by building the full unitary as a matrix product.

The circuit structure:
```
q0: ─Ry(θ₀)──●──Ry(θ₂)──●──
 │ │
q1: ─Ry(θ₁)──X──Ry(θ₃)──X──
```

In [ ]:
def build_vqe_ansatz_state(theta0, theta1, theta2, theta3):
    """Build the ansatz state |psi(theta)> as a 4-element vector.

    This constructs the state classically for use as a runtime input.
    In a real VQE, the uniqx would synthesise circuits from expv calls.
    """
    # Start with |00>
    psi = [1.0, 0.0, 0.0, 0.0]

    # Layer 1: Ry(theta0) on q0, Ry(theta1) on q1
    ry0 = ry_matrix(theta0)
    ry1 = ry_matrix(theta1)
    U1 = kron_2x2(ry0, ry1)
    psi = matvec(U1, psi)

    # CNOT
    cnot = cnot_matrix()
    psi = matvec(cnot, psi)

    # Layer 2: Ry(theta2) on q0, Ry(theta3) on q1
    ry2 = ry_matrix(theta2)
    ry3 = ry_matrix(theta3)
    U2 = kron_2x2(ry2, ry3)
    psi = matvec(U2, psi)

    # CNOT
    psi = matvec(cnot, psi)

    return psi


def matvec(M, v):
    """Matrix-vector product for small dense matrices."""
    n = len(v)
    return [sum(M[i][j] * v[j] for j in range(n)) for i in range(n)]

### 3.4 Energy evaluation via `ops.expect`

For each parameter setting, we:
1. Build the ansatz state vector classically (parameter preparation).
2. Send it to uniqx as a runtime input alongside the Hamiltonian.
3. uniqx computes $\langle\psi|H|\psi\rangle$ via `ops.expect`.

This is the **quantum** part of the hybrid loop, executed server-side.

In [ ]:
@tracing.to_module
def vqe_energy(H, psi):
    """Compute <psi|H|psi> — the variational energy."""
    energy = ops.expect(H, psi)
    return energy


# Trace the module once (defines the computation graph)
mod_vqe = vqe_energy(H2_hamiltonian, [1.0, 0.0, 0.0, 0.0])

print("VQE energy module traced.")
print(f"Module IR text (first 200 chars): {mod_vqe.to_text()[:200]}...")

### 3.5 The classical optimisation loop

Now we close the variational loop. The classical optimiser (here, a simple grid
search followed by refinement) calls uniqx repeatedly to evaluate the
energy at different parameter settings.

**The pattern:**
1. Classical: generate parameter values $\boldsymbol{\theta}$.
2. Classical: build the ansatz state $|\psi(\boldsymbol{\theta})\rangle$.
3. Server-side: compute $E(\boldsymbol{\theta}) = \langle\psi|H|\psi\rangle$ via uniqx.
4. Classical: update $\boldsymbol{\theta}$ based on $E$.
5. Repeat until convergence.

In [ ]:
def evaluate_energy(thetas, H_mat, module, client):
    """Evaluate VQE energy for given parameters via the uniqx.

    This function encapsulates one iteration of the variational loop:
    1. Build the ansatz state (classical, parameter-dependent).
    2. Submit to uniqx for energy evaluation (server-side).
    3. Parse and return the energy (classical).
    """
    # Step 1: Build the state vector from parameters
    psi = build_vqe_ansatz_state(*thetas)

    # Step 2: Submit to uniqx
    jid = submit(
        module,
        client=client,
        runtime_inputs=[H_mat, psi],
    )
    result = get(jid, client=client)

    # Step 3: Parse the result
    parsed = parse_result(result.get("result_payload", b""), ["energy"])
    energy = parsed["energy"][2][0]  # (dims, dtype, values) -> values[0]
    return energy

In [ ]:
# Parameter sweep: scan theta0 with other params fixed
# This demonstrates the outer classical loop
n_points = 12
theta_range = np.linspace(0, 2 * np.pi, n_points)
energies = []
valid_mask = []

print("Running VQE parameter sweep...")
for i, t0 in enumerate(theta_range):
    # Fix theta1=pi, theta2=0, theta3=0 for the 1D sweep
    try:
        E = evaluate_energy([t0, np.pi, 0.0, 0.0], H2_hamiltonian, mod_vqe, client)
        energies.append(E)
        valid_mask.append(True)
        if i % 4 == 0:
            print(f"  theta0={t0:.2f}, E={E:.6f} Ha")
    except (KeyError, IndexError, RuntimeError) as e:
        # Some compile-step paths still have known limitations on certain ops.
        print(f"  theta0={t0:.2f}: Known limitation: {type(e).__name__} — skipping point")
        energies.append(float("nan"))
        valid_mask.append(False)

valid_energies = [e for e, ok in zip(energies, valid_mask) if ok]
if valid_energies:
    best_idx = int(np.nanargmin(energies))
    print(f"\nBest energy from sweep: {energies[best_idx]:.6f} Ha")
    print(f"  at theta0 = {theta_range[best_idx]:.4f}")
else:
    print("\nNo valid energies returned in this sweep.")


### 3.6 Convergence plot

In [ ]:
# Filter out any failed iterations before plotting
plot_thetas = [t for t, e in zip(theta_range, energies) if not (isinstance(e, float) and e != e)]
plot_energies = [e for e in energies if not (isinstance(e, float) and e != e)]

if plot_energies:
    fig, ax = plt.subplots(figsize=(8, 5))

    ax.plot(plot_thetas, plot_energies, "o-", color="#2c3e50", markersize=4, label="VQE energy")
    ax.axhline(exact_energy, color="#e74c3c", linestyle="--", linewidth=1.5,
               label=f"Exact = {exact_energy:.4f} Ha")

    ax.set_xlabel(r"$\theta_0$ (radians)", fontsize=12)
    ax.set_ylabel("Energy (Hartree)", fontsize=12)
    ax.set_title(r"VQE Parameter Sweep: H$_2$ Ground State", fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Skipping VQE sweep plot: no valid energies to display.")


### 3.7 Full VQE with server-side state preparation

In the example above, we built the state vector classically and sent it to the
uniqx. For larger systems, the state vector is too large to construct classically.
Instead, we use `ops.expv` to prepare the state **server-side**:

$$|\psi(\theta)\rangle = e^{-i\theta G}\,|\psi_0\rangle$$

where $G$ is a generator (e.g. a Pauli term) and $\theta$ parameterises the rotation.
uniqx compiles this into a circuit or matrix exponential depending on hardware.

In [ ]:
@tracing.to_module
def vqe_expv(H, generator, psi0, theta):
    """VQE with server-side state preparation via expv.

    1. Prepare |psi(theta)> = exp(-i * theta * G) |psi0>   (server-side)
    2. Evaluate E = <psi|H|psi>                            (server-side)
    """
    # State preparation: apply parameterised unitary
    psi = ops.expv(generator, psi0, theta, hermitian=True)
    # Energy measurement
    energy = ops.expect(H, psi)
    return energy


# Build a generator: use the XX+YY part of the Hamiltonian as the ansatz generator
# This is physically motivated — it generates particle-hole excitations
G_matrix = add_mat(scale_mat(g4, H_XX), H_YY, g5)
psi0 = [1.0, 0.0, 0.0, 0.0]  # |00> initial state

# Trace the module
mod_expv = vqe_expv(H2_hamiltonian, G_matrix, psi0, 0.0)

# Sweep theta (reduced count for notebook runtime)
theta_vals = np.linspace(-np.pi, np.pi, 12)
energies_expv = []

print("Running VQE with expv state preparation...")
for i, t in enumerate(theta_vals):
    try:
        jid = submit(
            mod_expv,
            client=client,
            runtime_inputs=[H2_hamiltonian, G_matrix, psi0, t],
        )
        res = get(jid, client=client)
        parsed = parse_result(res.get("result_payload", b""), ["energy"])
        E = parsed["energy"][2][0]
        energies_expv.append(E)
        if i % 4 == 0:
            print(f"  theta={t:.2f}, E={E:.6f} Ha")
    except (KeyError, IndexError, RuntimeError) as e:
        # The expv lowering path has known limitations for some matrix shapes.
        print(f"  theta={t:.2f}: Known limitation: {type(e).__name__}")
        energies_expv.append(float("nan"))

valid_expv = [e for e in energies_expv if not (isinstance(e, float) and e != e)]
if valid_expv:
    best_idx_expv = int(np.nanargmin(energies_expv))
    print(f"\nBest energy (expv): {energies_expv[best_idx_expv]:.6f} Ha")
    print(f"Exact energy:       {exact_energy:.6f} Ha")
    print(f"Error:              {abs(energies_expv[best_idx_expv] - exact_energy):.6f} Ha")
else:
    print("\nexpv path returned no valid samples in this run (known limitation).")


---
## 4. QAOA Example: MaxCut on a Small Graph

The **Quantum Approximate Optimisation Algorithm** (QAOA) solves combinatorial
optimisation problems by alternating between a **cost unitary** and a **mixer
unitary**:

$$|\boldsymbol{\gamma}, \boldsymbol{\beta}\rangle = \prod_{p=1}^{P} e^{-i\beta_p B}\, e^{-i\gamma_p C} \;|+\rangle^{\otimes n}$$

where:
- $C$ is the **cost Hamiltonian** encoding the problem (e.g. MaxCut).
- $B = \sum_j X_j$ is the **mixer Hamiltonian** (transverse field).
- $\gamma_p, \beta_p$ are variational parameters.
- $P$ is the circuit depth (number of QAOA layers).

We demonstrate QAOA on the **MaxCut** problem: given a graph, partition vertices
into two sets to maximise the number of edges crossing the partition.

### 4.1 Define the graph

In [ ]:
# 4-node graph: a square with one diagonal
#   0 --- 1
#   |   / |
#   |  /  |
#   3 --- 2
#
# Edges: (0,1), (1,2), (2,3), (0,3), (1,3)
# Maximum cut = 4 (e.g. partition {0,2} vs {1,3})

n_qubits = 4
dim = 2 ** n_qubits  # 16-dimensional Hilbert space

edges = [(0, 1), (1, 2), (2, 3), (0, 3), (1, 3)]
max_cut_value = 4  # known optimal

print(f"Graph: {n_qubits} nodes, {len(edges)} edges")
print(f"Edges: {edges}")
print(f"Optimal MaxCut: {max_cut_value}")

### 4.2 Cost Hamiltonian

The MaxCut cost Hamiltonian is:

$$C = \sum_{(i,j) \in E} \frac{1}{2}(I - Z_i Z_j)$$

Each edge contributes $+1$ when the connected qubits are in different states
(one in $|0\rangle$, one in $|1\rangle$), and $0$ when they match.
The eigenvalues of $C$ are the cut sizes.

In [ ]:
def build_cost_hamiltonian(n_qubits, edges):
    """Build the MaxCut cost Hamiltonian as a dense matrix.

    C = sum_{(i,j) in edges} 0.5 * (I - Z_i Z_j)
    """
    dim = 2 ** n_qubits
    C = [[0.0] * dim for _ in range(dim)]

    for qi, qj in edges:
        # Build Z_i Z_j in the full Hilbert space
        ZZ = [[0.0] * dim for _ in range(dim)]
        II = [[0.0] * dim for _ in range(dim)]

        for k in range(dim):
            # Z eigenvalue: +1 if bit is 0, -1 if bit is 1
            bit_i = (k >> (n_qubits - 1 - qi)) & 1
            bit_j = (k >> (n_qubits - 1 - qj)) & 1
            zi = 1.0 - 2.0 * bit_i
            zj = 1.0 - 2.0 * bit_j
            ZZ[k][k] = zi * zj
            II[k][k] = 1.0

        # C += 0.5 * (I - ZZ)
        for k in range(dim):
            C[k][k] += 0.5 * (II[k][k] - ZZ[k][k])

    return C


C_ham = build_cost_hamiltonian(n_qubits, edges)

# Verify: eigenvalues should include the max cut value
C_np = np.array(C_ham)
eigvals_C = np.sort(np.linalg.eigvalsh(C_np))
print(f"Cost Hamiltonian eigenvalues: {np.unique(np.round(eigvals_C, 4))}")
print(f"Maximum eigenvalue (= max cut): {eigvals_C[-1]:.0f}")

### 4.3 Mixer Hamiltonian

The standard QAOA mixer is the transverse-field Hamiltonian:

$$B = \sum_{j=0}^{n-1} X_j$$

where $X_j$ is the Pauli-X operator on qubit $j$.

In [ ]:
def build_mixer_hamiltonian(n_qubits):
    """Build the transverse-field mixer B = sum_j X_j."""
    dim = 2 ** n_qubits
    B = [[0.0] * dim for _ in range(dim)]

    for q in range(n_qubits):
        # X_q flips bit q: |...0_q...> <-> |...1_q...>
        for k in range(dim):
            flipped = k ^ (1 << (n_qubits - 1 - q))
            B[k][flipped] += 1.0

    return B


B_ham = build_mixer_hamiltonian(n_qubits)
print(f"Mixer Hamiltonian: {n_qubits}-qubit transverse field")
print(f"  Matrix size: {dim}x{dim}")

### 4.4 QAOA circuit via `ops.expv`

The QAOA state is prepared by alternating cost and mixer unitaries:

$$|\gamma,\beta\rangle = e^{-i\beta B}\, e^{-i\gamma C}\, |+\rangle^{\otimes n}$$

Each `expv` call computes $\exp(-i t H) |\psi\rangle$ on uniqx. For QAOA
depth $p > 1$, we chain multiple `expv` calls.

In [ ]:
@tracing.to_module
def qaoa_energy(C, B, psi_plus, gamma, beta):
    """QAOA depth-1: prepare state and measure cost.

    Steps (all server-side):
      1. Apply cost unitary:  |psi1> = exp(-i * gamma * C) |+>
      2. Apply mixer unitary: |psi2> = exp(-i * beta * B) |psi1>
      3. Measure cost: E = <psi2| C |psi2>
    """
    # Cost unitary
    psi1 = ops.expv(C, psi_plus, gamma, hermitian=True)
    # Mixer unitary
    psi2 = ops.expv(B, psi1, beta, hermitian=True)
    # Measure
    cost = ops.expect(C, psi2)
    return cost


# Initial state: |+>^n = H^n |0>^n = (1/sqrt(2^n)) * [1,1,...,1]
psi_plus = [1.0 / math.sqrt(dim)] * dim

# Trace the module
mod_qaoa = qaoa_energy(C_ham, B_ham, psi_plus, 0.0, 0.0)
print("QAOA depth-1 module traced.")

### 4.5 Parameter landscape

QAOA depth-1 has two parameters: $\gamma$ (cost) and $\beta$ (mixer). We sweep
both to visualise the energy landscape and find the optimal parameters.

In [ ]:
# 2D parameter sweep (reduced grid for notebook runtime)
n_gamma = 6
n_beta = 6
gamma_range = np.linspace(0, np.pi, n_gamma)
beta_range = np.linspace(0, np.pi, n_beta)

landscape = np.full((n_gamma, n_beta), np.nan)

print("Scanning QAOA parameter landscape...")
for i, g in enumerate(gamma_range):
    for j, b in enumerate(beta_range):
        try:
            jid = submit(
                mod_qaoa,
                client=client,
                runtime_inputs=[C_ham, B_ham, psi_plus, float(g), float(b)],
            )
            res = get(jid, client=client)
            parsed = parse_result(res.get("result_payload", b""), ["cost"])
            landscape[i, j] = parsed["cost"][2][0]
        except (KeyError, IndexError, RuntimeError) as e:
            # expv on 16-dim matrices can hit known compile-step limitations.
            landscape[i, j] = float("nan")
    if i % 2 == 0:
        print(f"  gamma row {i}/{n_gamma} done")

if np.all(np.isnan(landscape)):
    print("\nQAOA landscape: Known limitation — no valid samples.")
    best_gamma = float(gamma_range[0])
    best_beta = float(beta_range[0])
    best_cost = 0.0
else:
    best_ij = np.unravel_index(np.nanargmax(landscape), landscape.shape)
    best_gamma = gamma_range[best_ij[0]]
    best_beta = beta_range[best_ij[1]]
    best_cost = landscape[best_ij]
    print(f"\nBest QAOA cost: {best_cost:.4f}")
    print(f"  gamma* = {best_gamma:.4f}, beta* = {best_beta:.4f}")
    print(f"  Optimal MaxCut = {max_cut_value} (ratio: {best_cost / max_cut_value:.3f})")


In [ ]:
# Plot the landscape
if np.all(np.isnan(landscape)):
    print("Skipping landscape plot: no valid samples to display.")
else:
    fig, ax = plt.subplots(figsize=(8, 6))

    im = ax.imshow(
        landscape.T,
        extent=[gamma_range[0], gamma_range[-1], beta_range[0], beta_range[-1]],
        origin="lower",
        aspect="auto",
        cmap="viridis",
    )
    ax.plot(best_gamma, best_beta, "r*", markersize=15, label=f"Optimum = {best_cost:.2f}")

    ax.set_xlabel(r"$\gamma$", fontsize=14)
    ax.set_ylabel(r"$\beta$", fontsize=14)
    ax.set_title("QAOA Depth-1 Energy Landscape (MaxCut)", fontsize=14)
    ax.legend(fontsize=12)
    plt.colorbar(im, ax=ax, label="Expected cut value")
    plt.tight_layout()
    plt.show()


### 4.6 Improving solution quality with circuit depth $p$

QAOA at depth $p=1$ gives an approximation ratio that depends on the graph.
Increasing $p$ improves the approximation. At depth $p$, there are $2p$ parameters
$(\gamma_1, \beta_1, \ldots, \gamma_p, \beta_p)$.

We trace a depth-$p$ module by chaining `expv` calls in a loop.

In [ ]:
def qaoa_depth_p(C_mat, B_mat, psi_init, gammas, betas, p):
    """Evaluate QAOA at depth p with given parameters.

    For each layer, we trace and submit a separate expv+expect module.
    The state is evolved layer by layer on uniqx.
    """
    psi = list(psi_init)  # current state

    for layer in range(p):
        # Trace a single-layer evolution module
        @tracing.to_module
        def evolve_layer(C, B, psi_in, gamma_l, beta_l):
            psi1 = ops.expv(C, psi_in, gamma_l, hermitian=True)
            psi2 = ops.expv(B, psi1, beta_l, hermitian=True)
            return psi2

        try:
            mod_layer = evolve_layer(C_mat, B_mat, psi, gammas[layer], betas[layer])
            jid = submit(
                mod_layer,
                client=client,
                runtime_inputs=[C_mat, B_mat, psi, float(gammas[layer]), float(betas[layer])],
            )
            res = get(jid, client=client)
            parsed = parse_result(res.get("result_payload", b""), ["psi_out"])
            psi = parsed["psi_out"][2]  # updated state vector
        except (KeyError, IndexError, RuntimeError):
            return float("nan")

    # Final measurement
    @tracing.to_module
    def measure_cost(C, psi_final):
        return ops.expect(C, psi_final)

    try:
        mod_measure = measure_cost(C_mat, psi)
        jid = submit(mod_measure, client=client, runtime_inputs=[C_mat, psi])
        res = get(jid, client=client)
        parsed = parse_result(res.get("result_payload", b""), ["cost"])
        return parsed["cost"][2][0]
    except (KeyError, IndexError, RuntimeError):
        return float("nan")


# Compare p=1,2 (depth=3 dropped to keep notebook runtime short)
print("QAOA quality vs circuit depth:")
print(f"{'Depth p':<10} {'Best cost':<15} {'Ratio':<10}")
print("-" * 35)

# Use the optimal p=1 parameters as a starting point
for p in [1, 2]:
    # Heuristic: repeat the best p=1 parameters for each layer
    gammas = [best_gamma] * p
    betas = [best_beta] * p
    cost = qaoa_depth_p(C_ham, B_ham, psi_plus, gammas, betas, p)
    if isinstance(cost, float) and cost != cost:
        print(f"{p:<10} {'Known limitation':<15} {'-':<10}")
    else:
        print(f"{p:<10} {cost:<15.4f} {cost / max_cut_value:<10.3f}")


---
## 5. Result Conversion: Parsing uniqx Results

uniqx returns results as **buffer-view** strings in a binary payload.
Understanding how to parse these back into Python arrays is essential for
closing the classical-quantum loop.

### 5.1 Buffer-view format

Each result line has the format:
```
<dim1>x<dim2>x...x<dtype>= <value1> <value2>...
```

Examples:
- Scalar: `f64= -1.137`
- Vector: `4xf64= 1.0 0.0 0.0 0.0`
- Matrix: `4x4xf64= 1.0 0.0 0.0 0.0 0.0 1.0...` (row-major)

### 5.2 Using `parse_buffer_view`

In [ ]:
# Demonstrate parse_buffer_view on example strings
examples = [
    "f64= -1.137",
    "4xf64= 1.0 0.0 0.0 0.0",
    "2x2xf64= 1.0 0.0 0.0 -1.0",
    "3xi64= 0 1 2",
]

for ex in examples:
    result = parse_buffer_view(ex)
    dims, dtype, values = result
    print(f"Input:  '{ex}'")
    print(f"  dims={dims}, dtype='{dtype}', values={values}")
    if len(dims) == 2:
        rows, cols = dims
        matrix = [values[i * cols:(i + 1) * cols] for i in range(rows)]
        print(f"  As matrix: {matrix}")
    print()

### 5.3 Using `parse_result`

For multi-output modules, `parse_result` maps each output to a named entry.
You provide a list of names matching the module's output order.

In [ ]:
# Example: a module that returns both eigenvalues and the energy
@tracing.to_module
def multi_output(H, psi):
    """Return both energy and full eigenspectrum."""
    energy = ops.expect(H, psi)
    eigvals, eigvecs = ops.eigs(H, k=4, which="smallest", hermitian=True)
    return energy, eigvals


try:
    mod_multi = multi_output(H2_hamiltonian, [1.0, 0.0, 0.0, 0.0])
    jid = submit(
        mod_multi,
        client=client,
        runtime_inputs=[H2_hamiltonian, [1.0, 0.0, 0.0, 0.0]],
    )
    res = get(jid, client=client)

    # Parse with named outputs
    parsed = parse_result(
        res.get("result_payload", b""),
        ["energy", "eigenvalues"],
    )

    print("Multi-output result:")
    for name, (dims, dtype, values) in parsed.items():
        print(f"  {name}: dims={dims}, dtype={dtype}, values={values}")
except (KeyError, IndexError, RuntimeError) as e:
    print(f"Known limitation: multi-output eigs path ({type(e).__name__})")


### 5.4 Encoding runtime inputs

When submitting a module, runtime inputs must match the traced module's parameter
signature. The SDK handles encoding automatically via `runtime_inputs`:

- **Scalars**: `float` or `int` values map to `f64=` or `i64=`.
- **Vectors**: Python lists map to `NxDTYPE=` format.
- **Matrices**: Nested lists map to `MxNxDTYPE=` (row-major).

You can also encode manually using `fmt_mat`, `fmt_vec`, `fmt_scalar`:

In [ ]:
# Manual encoding examples
print("Manual buffer-view encoding:")
print(f"  Scalar: {fmt_scalar(-1.137)}")
print(f"  Vector: {fmt_vec([1.0, 0.0, 0.0, 0.0], 4)}")
print(f"  Matrix: {fmt_mat([[1, 0], [0, -1]], 2, 2)}")
print()
print("These are equivalent to passing Python objects via runtime_inputs.")
print("The SDK encodes them automatically when you pass lists/floats.")

---
## 6. Putting It All Together: The Full Variational Loop

Here is the complete pattern for a hybrid quantum-classical algorithm on uniqx:

```python
# 1. TRACE: Define the quantum computation as a module
@tracing.to_module
def quantum_cost(H, psi0, theta):
 psi = ops.expv(H, psi0, theta, hermitian=True) # state prep
 return ops.expect(H, psi) # measurement

module = quantum_cost(H_matrix, psi0, 0.0) # trace once

# 2. CONNECT: Establish uniqx connection
client = connect(os.environ.get("UNIQX_GATEWAY", "api.oriqx.com:443"))

# 3. LOOP: Classical optimiser drives the quantum evaluation
for iteration in range(max_iters):
 # 3a. Submit with current parameters
 job_id = submit(module, client=client,
 runtime_inputs=[H_matrix, psi0, theta])

 # 3b. Wait for result
 result = get(job_id, client=client)

 # 3c. Parse the quantum result back to a Python number
 parsed = parse_result(result["result_payload"], ["cost"])
 cost = parsed["cost"][2][0]

 # 3d. Classical update
 theta = optimizer.step(theta, cost)
```

This pattern works for VQE, QAOA, quantum machine learning, and any other
variational algorithm. uniqx handles routing to the right hardware
(CPU, GPU, or QPU) transparently.

### Key takeaways

1. **Trace once, execute many**: The `@tracing.to_module` decorator captures the
 computation graph. Re-execution with different `runtime_inputs` is cheap.

2. **All quantum ops are server-side**: `ops.expv`, `ops.expect`, and `ops.eigs`
 are traced into the IR and compiled/executed by uniqx. The Python side
 only orchestrates the classical loop.

3. **Parameters flow as runtime inputs**: The classical optimiser updates parameter
 values and passes them to uniqx on each iteration.

4. **Results flow back as buffer views**: uniqx returns results in a compact
 binary format. Use `parse_result` or `parse_buffer_view` to convert back to
 Python numbers.

5. **Hardware-agnostic**: The same code runs on CPU simulators, GPU accelerators,
 and quantum processors. uniqx handles routing and compilation.

---

*This notebook is part of the uniqx SDK examples. For more information,
see the [uniqx documentation](https://docs.oriqx.com).*